In [15]:
# Визначення DataFrame з тривимірними векторами з word embeddings та застосування PCA для зменшення розмірності до 3-х векторів

import pickle
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import gdown

# 1. Завантаження моделі word embeddings з Google Drive
file_id = '1281E0CDneuKdflWFBUvuyUzujpdGVImz'
url = f'https://drive.google.com/uc?id={file_id}'
output_file = 'word_embeddings_subset.p'
print("Завантаження файлу...")
gdown.download(url, output_file, quiet=False)

# Відкриваємо завантажений файл
print("\nОбробка даних...")
with open(output_file, 'rb') as f:
    word_embeddings = pickle.load(f)

# Створюємо списки для слів та векторів
words = list(word_embeddings.keys())
vectors = np.array(list(word_embeddings.values()))

# 2. Використовуємо PCA для зменшення розмірності до 3-х векторів
pca = PCA(n_components=3)
vectors_3d = pca.fit_transform(vectors)

# 3. Створюємо DataFrame
df = pd.DataFrame({
    'word': words,
    'x': vectors_3d[:, 0],
    'y': vectors_3d[:, 1],
    'z': vectors_3d[:, 2]
})

print("\nГотовий DataFrame:")
print(df.tail(50))

Завантаження файлу...


Downloading...
From: https://drive.google.com/uc?id=1281E0CDneuKdflWFBUvuyUzujpdGVImz
To: /content/word_embeddings_subset.p
100%|██████████| 309k/309k [00:00<00:00, 60.6MB/s]


Обробка даних...

Готовий DataFrame:
             word         x         y         z
193          Suva -1.342464 -0.848763 -0.890645
194       Yerevan -1.057712  1.580594  1.107399
195        Zagreb -0.263384  1.756496  0.277361
196       Bishkek -1.513941  1.361890  0.874304
197        Manama -1.213770  0.059522 -0.602088
198        Kigali -1.678399 -0.704023  0.715195
199          Riga -0.314410  1.700821  0.120806
200        Lusaka -1.808256 -1.109714  0.550852
201      Tashkent -0.771772  1.287002  0.481882
202       Nicosia -0.623296  0.935671 -0.168360
203      Valletta -0.510956  0.362901 -0.022450
204      Windhoek -1.516317 -0.556969  0.450443
205      Dominica  0.745012 -0.938147 -0.404262
206         Quito -0.760938  0.082956 -0.806365
207       Tallinn -0.502376  1.958417 -0.021230
208    Bratislava -0.471912  1.834467  0.103672
209   Tegucigalpa -1.593014 -0.165771 -0.659035
210        Skopje -0.488989  1.945863  0.836101
211      Gaborone -1.511013 -1.119713  0.502368
21

In [18]:


# Функція для отримання вектора конкретного слова з DataFrame
def get_vector(word, df):
    return df[df['word'] == word][['x', 'y', 'z']].values[0]

# Функція для пошуку найближчого слова
def find_nearest_word(target_vector, df):
    # Перетворюємо вхідний вектор на numpy масив
    target_vector = np.array(target_vector)
    
    # Витягуємо всі 3D координати з датафрейму
    all_vectors = df[['x', 'y', 'z']].values
    
    # Знаходимо відстані від нашого вектора до всіх інших
    distances = np.linalg.norm(all_vectors - target_vector, axis=1)
    
    # Шукаємо індекс найменшої відстані
    nearest_idx = np.argmin(distances)
    
    return df.iloc[nearest_idx]['word'], distances[nearest_idx]

# --- ТЕСТУВАННЯ ---
print("--- Тестування пошуку найближчого слова ---")

# 1. Точний збіг
vec_country = get_vector('Bratislava', df)
word, dist = find_nearest_word(vec_country, df)
print(f"Ціль: 'Bratislava'. Знайдено: '{word}', відстань: {dist:.4f}")

# 2. Трохи змінений вектор (додаємо шум)
noisy_vec = vec_country + np.array([0.2, -0.02, 0.05])
word_noisy, dist_noisy = find_nearest_word(noisy_vec, df)
print(f"Ціль: 'Bratislava' + шум. Знайдено: '{word_noisy}', відстань: {dist_noisy:.4f}")

--- Тестування пошуку найближчого слова ---
Ціль: 'Bratislava'. Знайдено: 'Bratislava', відстань: 0.0000
Ціль: 'Bratislava' + шум. Знайдено: 'Riga', відстань: 0.1257


In [21]:
import plotly.graph_objects as go
import numpy as np

def visualize_search_3d(target_vector, target_word, nearest_word, df):
    """
    Візуалізує простір слів у 3D, виділяючи цільовий вектор та знайдений результат.
    """
    fig = go.Figure()

    # 1. Малюємо всі інші слова (фон) сірим кольором
    other_words = df[(df['word'] != target_word) & (df['word'] != nearest_word)]
    fig.add_trace(go.Scatter3d(
        x=other_words['x'], y=other_words['y'], z=other_words['z'],
        mode='markers',
        text=other_words['word'], # Текст при наведенні
        marker=dict(size=4, color='lightgrey', opacity=0.6),
        name='Інші слова'
    ))

    # 2. Малюємо ЦІЛЬОВИЙ ВЕКТОР (наприклад, слово або середнє значення) червоним
    if target_word:
        target_name = f"Ціль: {target_word}"
    else:
        target_name = "Цільовий вектор (координати)"
        
    fig.add_trace(go.Scatter3d(
        x=[target_vector[0]], y=[target_vector[1]], z=[target_vector[2]],
        mode='markers+text',
        text=[target_name],
        textposition="top center",
        marker=dict(size=8, color='red', symbol='diamond'),
        name='Ціль'
    ))

    # 3. Малюємо ЗНАЙДЕНЕ НАЙБЛИЖЧЕ СЛОВО зеленим
    nearest_data = df[df['word'] == nearest_word]
    fig.add_trace(go.Scatter3d(
        x=nearest_data['x'], y=nearest_data['y'], z=nearest_data['z'],
        mode='markers+text',
        text=[f"Знайдено: {nearest_word}"],
        textposition="bottom center",
        marker=dict(size=8, color='green'),
        name='Найближче слово'
    ))

    # 4. З'єднуємо їх лінією для наочності
    fig.add_trace(go.Scatter3d(
        x=[target_vector[0], nearest_data['x'].values[0]],
        y=[target_vector[1], nearest_data['y'].values[0]],
        z=[target_vector[2], nearest_data['z'].values[0]],
        mode='lines',
        line=dict(color='blue', width=2, dash='dash'),
        name='Відстань'
    ))

    # Налаштування вигляду графіка
    fig.update_layout(
        title="3D Візуалізація Word Embeddings",
        scene=dict(xaxis_title='PCA 1 (X)', yaxis_title='PCA 2 (Y)', zaxis_title='PCA 3 (Z)'),
        width=900, height=700,
        showlegend=True
    )
    
    fig.show()

# ==========================================
# ПРИКЛАД ВИКОРИСТАННЯ:
# ==========================================

# Візьмемо для прикладу усереднений вектор між 'France' та 'Germany'
vec_france = df[df['word'] == 'France'][['x', 'y', 'z']].values[0]
vec_germany = df[df['word'] == 'Germany'][['x', 'y', 'z']].values[0]

target_vec = (vec_france + vec_germany) / 2
found_word, dist = find_nearest_word(target_vec, df)

# Викликаємо візуалізацію
visualize_search_3d(
    target_vector=target_vec, 
    target_word="Середнє (France + Germany)", 
    nearest_word=found_word, 
    df=df
)